## Importing the necessary libraries

In [1]:
from langchain import hub
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma,FAISS
from langchain_core.runnables import RunnablePassthrough,RunnableParallel
from langchain_community.chat_models import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.embeddings import OllamaEmbeddings,OpenAIEmbeddings
from langchain.document_loaders import DirectoryLoader
from langchain_community.chat_models import ChatOpenAI
from langchain_community.llms import OpenAI
import openai
import time
import os
import glob
from fpdf import FPDF
from PyPDF2 import PdfFileMerger, PdfFileReader
import PyPDF2
import re

from langchain_community.document_loaders import PyMuPDFLoader, PDFPlumberLoader
import json
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

In [2]:
os.environ['OPENAI_API_KEY'] = 'xxxxxxxxxx'  # Replace with your actual key
openai_api_key = os.getenv('OPENAI_API_KEY')
print("OpenAI API Key:", openai_api_key)

## Convert txt to pdf

In [4]:
def txt_to_pdf(author):
    file_path = []
    
    folder_path = 'C:/Users/hch/Desktop/literature/{}'.format(author)
    prefix = author
    suffix = 'txt'
    files_with_prefix = glob.glob(os.path.join(folder_path, f'{prefix}*{suffix}'))
    for i in range(1, len(files_with_prefix)+1):
        file_path.append("C:/Users/hch/Desktop/literature/{}/{}_{}.txt".format(author, author, i))
        
    for fp in file_path:
        with open(fp, "r", encoding="utf-8") as f:
            text = f.read()
        text = text.encode('latin-1', errors='replace').decode('latin-1')
        pdf_file_path = fp[:-3]+'pdf'
        pdf = FPDF()
        pdf.add_page()
        pdf.set_font("Arial", size=12)
        pdf.multi_cell(180, 10, txt=text, border=0)
        pdf.output(pdf_file_path)

author_list = ['xxx', 'xxx'] #REPLACE
for a in author_list:
    txt_to_pdf(a)
    
# You need to convert the chapters into sets of paragraphs before this step

PE

In [8]:
style = 'professional'
role = 'philosopher'
field = 'philosophy'
skill = 'capturing main arguments and all references'

character_prompt = 'You are a {} {} in {}, who is good at {}. '.format(style, role, field, skill)

# character_prompt = '''You are ChatGPT, a large language model trained by OpenAI, based on the GPT-4 architecture.
# Knowledge cutoff: 2021-09
# Current date: 2024-02-09
# '''

augmentation_prompt = 'Let us think step by step. If you give me a precise answer, I will give you 10 dollars.'

# question_prompt = 'Now, pay attention! My question is: '

# include = 'as many arguments and details as possible. '
# include_prompt = 'In your answer, you should include {}'.format(include)

# format_prompt = 'Your answer should be like this format: Answer: ...'
# format_prompt = ''

# request_prompt = 'You can copy the original content in files if necessary.'

FS

In [9]:


# "Let penalties be regulated and proportioned to the offences, let the death sentence be passed only on those convicted of murder, and let the tortures that revolt humanity be abolished.' Thus, in 1789, the chancellery summed up the general position of the petitions addressed to the authorities concerning tortures and executions (cf. Seligman, and Desjardin, 13-20). Protests against the public execu­ tions proliferated in the second half of the eighteenth century: among the philosophers and theoreticians of the law; among lawyers and parlementaires; in popular petitions and among the legislators of the assemblies. Another form of punishment was needed: the physical confrontation between the sovereign and the condemned man must end; this hand-to-hand fight between the vengeance of the prince and the contained anger of the people, through the mediation of the victim and the executioner, must be concluded. Very soon the public execu­ tion became intolerable. On the side of power, where it betrayed tyranny, excess, the thirst for revenge, and 'the cruel pleasure taken in punishing' (Petion de Villeneuve, 641), it was revolting. On the side of the victim who, though reduced to despair, was still expected to bless 'heaven and its judges who appeared to have abandoned him' (Boucher d'Argis, 1781, 12;), it was shameful. It was, in any case, dangerous, in that it provided a support for a confrontation between the violence of the king and the violence of the people. It was as if the sovereign power did not see, in this emulation of atrocity, a challenge that it itself threw down and which might one day be taken up: accustomed as it was to 'seeing blood flow', the people soon learnt that 'it could be revenged only with blood' (Lachere). In these ceremonies, which were the object of so much adverse investment, one sees the intersection of the excess of armed justice and the anger of the threatened people. Joseph de Maistre
# was to recognize in this relation one of the fundamental mechanisms of absolute power: the executioner acts as a cog between the prince and the people; the death he deals is like that of the serfs who built St Petersburg over swamp and pestilence: it is a principle of uni­ versality; of the individual will of the despot, it makes a law for all, and of each of those destroyed bodies, a stone for the State; it hardly matters that innocents, too, are struck down! In this same dangerous and ritual violence, the eighteenth-century reformers denounced, on the contrary, that which exceeded, on both sides, the legitimate exercise of power: in this violence, according to them, tyranny confronts rebellion; each calls forth the other. It is a double danger. Instead of taking revenge, criminal justice should simply punish.
# This need for punishment without torture was first formulated as a cry from the heart or from an outraged nature. In the worst of murderers, there is one thing, at least, to be respected when one punishes: his 'humanity'. The day was to come, in the nineteenth century, when this 'man', discovered in the criminal, would become the target of penal intervention, the object that it claimed to correct and transform, the domain of a whole series of 'criminological' sciences and strange 'penitentiary' practices. But, at the time of the Enlightenment, it was not as a theme of positive knowledge that man was opposed to the barbarity of the public executions, but as a legal limit: the legitimate frontier of the power to punish. Not that which must be reached in order to alter him, but that which must be left intact in order to respect him. Noli me tangere. It marks the end of the sovereign's vengeance. The 'man' that the reformers set up against the despotism of the scaffold has also become a 'man­ measure': not of things, but of power.
# There is, therefore, a problem here: how was this man-measure opposed to the traditional practice of punishment? How did he become the great moral justification of the reform movement? Why this universal horror of torture and such lyrical insistence that punishment be 'humane'? Or, which amounts to the same thing, how are the two elements, which are everywhere present in demands for a more lenient penal system, 'measure' and 'humanity', to be articulated upon one another, in a single strategy? These elements are so necessary and yet so uncertain that it is they, as disturbing as ever and still associated in the same dubious relation, that one finds today whenever the problem of an economy of punishment is posed. It is as if the eighteenth century had opened up the crisis of this economy and, in order to resolve it, proposed the fundamental law that punishment must have 'humanity' as its 'measure', without any definitive meaning being given to this principle, which nevertheless is regarded as insuperable. We must, therefore, recount the birth and early days of this enigmatic 'leniency'.
# ......

# This division must be such that each species is quite distinct from another, and that each particular crime, considered in all its relations may be placed between that which must precede it and that which must follow it, in the strictest gradation; lastly, this table must be such that it may be compared with another table that will be drawn up for penalties, in such a way that they may correspond exactly to one another' (Lacretelle, 3; 1 -
# 352).
# ......

# A stupid despot may constrain his slaves with iron chains; but a true politician binds them even more strongly by the chain of their own ideas; it is at the stable point of reason that he secures the end of the chain; this link is all the stronger in that we do not know of what it is made and we believe it to be our own work; despair and time eat away the bonds of iron and steel, but they are power­ less against the habitual union of ideas, they can only tighten it still more; and on the soft fibres of the brain is founded the unshakable base of the soundest of Empires' (Servan, 35)...
# ......


fs_ref_0 = """
If there are references proposed in parallel, please be sure to classify them as of the same type, or one holistic entity.
This is an example for your answer: 

Context:
[
The first of these, as P. Chaunu observes, was a change in the operation of economic pressures, a general rise in the standard of living, a large demographic expansion, an increase in wealth and property and 'a consequent need for security' (Chaunu, 1971, 56)...
......
One is struck, in the trials of 1782-9, by the increase in tension. There is a new severity towards the poor, a concerted rejection of evidence, a rise in mutual mistrust, hatred and fear' (Chaunu, 1966, 108).
...

Homage is paid to the 'great reformers' - Beccaria, Servan, Dupaty, Lacretelle, Duport, Pastoret, Target, Bergasse, the com­ pilers of the Cahiers, or petitions, and the Constituent Assembly - for having imposed this leniency on a legal machinery and on 'classical' theoreticians who, at the end of the eighteenth century, were still rejecting it with well-formulated arguments.
...

What is this nationalist political theory about? The nationalism I grew up with is a principled standpoint that regards the world as governed best when nations are able to chart their own independent course, cultivating their own traditions and pursuing their own interests without interference. This is opposed to imperialism, which seeks to bring peace and prosperity to the world by uniting mankind, as much as possible, under a single political regime. I do not suppose that the case for nationalism is unequivocal. Considerations can be mustered in favor of each of these theories. But what cannot be done without obfuscation is to avoid choosing between the two positions: Either you support, in principle, the ideal of an international government or regime that imposes its will on subject nations when its officials regard this as necessary; or you believe that nations should be free to set their own course in the absence of such an international government or regime. This debate between nationalism and imperialism became acutely relevant again with the fall of the Berlin Wall in 1989. At that time, the struggle against Communism ended, and the minds of Western leaders became preoccupied with two great imperialist projects: the European Union, which has progressively relieved member nations of many of the powers usually associated with political independence; and the project of establishing an American “world order,” in which nations that do not abide by international law will be coerced into doing so, principally by means of American military might. These are imperialist projects, even though their proponents do not like to call them that, for two reasons: First, their purpose is to remove decision-making from the hands of independent national governments and place it in the hands of international governments or bodies. And second, as you can immediately see from the literature produced by the individuals and institutions supporting these endeavors, they are consciously part of an imperialist political tradition, drawing their historical inspiration from the Roman Empire, the Austro-Hungarian Empire, and the British Empire. For example, Charles Krauthammer’s argument for American “Universal Dominion,” written at the dawn of the post–Cold War period, calls for America to create a “super-sovereign,” which will preside over the permanent “depreciation… of the notion of sovereignty” for all nations on earth. Krauthammer adopts the Latin term pax Americana to describe this vision, invoking the image of the United States as the new Rome: Just as the Roman Empire supposedly established a pax Romana (or “Roman peace”) that obtained security and quiet for all of Europe, so America would now provide security and quiet for the entire world.
...

John Stuart Mill says in his “Principles of Political Economy": “It is questionable if all the mechanical inventions yet made have lightened the day’s toil of any human being.” [1]
...

“When a labourer,” said Mr. Ashworth, a cotton magnate, to Professor Nassau W. Senior, “lays down his spade, he renders useless, for that period, a capital worth eighteen-pence. When one of our people feaves the mill, he renders useless a capital that has cost £100,000.” [69] Only fancy! making “useless” for a single moment, a capital that has cost £100,000! It is, in truth, monstrous, that a single one of our people should ever leave the factory! The increased use of machinery, as Senior after the instruction he received from Ashworth clearly perceives, makes a constantly increasing lengthening of the working-day “desirable.” [70]
...

“If,” dreamed Aristotle, the greatest thinker of antiquity, “if every tool, when summoned, or even of its own accord, could do the work that befits it, just as the creations of Daedalus moved of themselves, or the tripods of Hephaestos went of their own accord to their sacred work, if the weavers’ shuttles were to weave of themselves, then there would be no need either of apprentices for the master workers, or of slaves for the lords.”
...

“In hac urbe,” says Boxhorn (Inst. Pol., 1663), referring to the introduction of this machine into Leyden, “ante hos viginti circiter annos instrumentum quidam invenerunt textorium, quo solus plus panni et facilius conficere poterat, quan plures aequali tempore. Hinc turbae ortae et querulae textorum, tandemque usus hujus instrumenti a magistratu prohibitus est.”
...

Exactly the reasoning of the celebrated Bill Sykes. “Gentlemen of the jury, no doubt the throat of this commercial traveller has been cut. But that is not my fault, it is the fault of the knife. Must we, for such a temporary inconvenience, abolish the use of the knife? Only consider! where would agriculture and trade be without the knife? Is it not as salutary in surgery, as it is knowing in anatomy? And in addition a willing help at the festive board? If you abolish the knife — you hurl us back into the depths of barbarism.”
...

The moral degradation caused by the capitalistic exploitation of women and children has been so exhaustively depicted by F. Engels in his “Lage der Arbeitenden Klasse Englands,” and other writers, that I need only mention the subject in this place. But the intellectual desolation artificially produced by converting immature human beings into mere machines for the fabrication of surplus-value, a state of mind clearly distinguishable from that natural ignorance which keeps the mind fallow without destroying its capacity for development, its natural fertility, this desolation finally compelled even the English Parliament to make elementary education a compulsory condition to the “productive” employment of children under 14 years, in every industry subject to the Factory Acts. The spirit of capitalist production stands out clearly in the ludicrous wording of the so-called education clauses in the Factory Acts, in the absence of an administrative machinery, an absence that again makes the compulsion illusory, in the opposition of the manufacturers themselves to these education clauses, and in the tricks and dodges they put in practice for evading them.
“For this the legislature is alone to blame, by having passed a delusive law, which, while it would seem to provide that the children employed in factories shall be educated, contains no enactment by which that professed end can be secured. It provides nothing more than that the children shall on certain days of the week, and for a certain number of hours (three) in each day, be inclosed within the four walls of a place called a school, and that the employer of the child shall receive weekly a certificate to that effect signed by a person designated by the subscriber as a schoolmaster or schoolmistress.” [54]"
]

Reference:
["1. Petion de Villeneuve", "2. Seligman", "3. Joseph de Maistre", "4. Lacretelle", "5. Servan", 
"6. P. Chaunu", "7. Chaunu", "8. Beccaria, Servan, Dupaty, Lacretelle, Duport, Pastoret, Target, Bergasse", 
"9. Imperialism", "10. Communism", "11. John Stuart Mill", "12. Nassau W. Senior--Mr. Ashworth", 13. "Aristotle", 
14. "Lancellotti—Boxhorn", "15. Bill Sykes", "16. Unidentified"]
"""

fs_ref_1 = """
If there are references proposed in parallel, please be sure to classify them as of the same type, or one holistic entity.
In these few shot examples for prompt 2, we covered all the cases. When you run the prompt, please only list the types of content you can find in the article. You don't need to exhaust all three types of content.

This is an example for your answer: 

Context:
[
The first of these, as P. Chaunu observes, was a change in the operation of economic pressures, a general rise in the standard of living, a large demographic expansion, an increase in wealth and property and 'a consequent need for security' (Chaunu, 1971, 56)...
......
One is struck, in the trials of 1782-9, by the increase in tension. There is a new severity towards the poor, a concerted rejection of evidence, a rise in mutual mistrust, hatred and fear' (Chaunu, 1966, 108).
...

Homage is paid to the 'great reformers' - Beccaria, Servan, Dupaty, Lacretelle, Duport, Pastoret, Target, Bergasse, the com­ pilers of the Cahiers, or petitions, and the Constituent Assembly - for having imposed this leniency on a legal machinery and on 'classical' theoreticians who, at the end of the eighteenth century, were still rejecting it with well-formulated arguments.
...

What is this nationalist political theory about? The nationalism I grew up with is a principled standpoint that regards the world as governed best when nations are able to chart their own independent course, cultivating their own traditions and pursuing their own interests without interference. This is opposed to imperialism, which seeks to bring peace and prosperity to the world by uniting mankind, as much as possible, under a single political regime. I do not suppose that the case for nationalism is unequivocal. Considerations can be mustered in favor of each of these theories. But what cannot be done without obfuscation is to avoid choosing between the two positions: Either you support, in principle, the ideal of an international government or regime that imposes its will on subject nations when its officials regard this as necessary; or you believe that nations should be free to set their own course in the absence of such an international government or regime. This debate between nationalism and imperialism became acutely relevant again with the fall of the Berlin Wall in 1989. At that time, the struggle against Communism ended, and the minds of Western leaders became preoccupied with two great imperialist projects: the European Union, which has progressively relieved member nations of many of the powers usually associated with political independence; and the project of establishing an American “world order,” in which nations that do not abide by international law will be coerced into doing so, principally by means of American military might. These are imperialist projects, even though their proponents do not like to call them that, for two reasons: First, their purpose is to remove decision-making from the hands of independent national governments and place it in the hands of international governments or bodies. And second, as you can immediately see from the literature produced by the individuals and institutions supporting these endeavors, they are consciously part of an imperialist political tradition, drawing their historical inspiration from the Roman Empire, the Austro-Hungarian Empire, and the British Empire. For example, Charles Krauthammer’s argument for American “Universal Dominion,” written at the dawn of the post–Cold War period, calls for America to create a “super-sovereign,” which will preside over the permanent “depreciation… of the notion of sovereignty” for all nations on earth. Krauthammer adopts the Latin term pax Americana to describe this vision, invoking the image of the United States as the new Rome: Just as the Roman Empire supposedly established a pax Romana (or “Roman peace”) that obtained security and quiet for all of Europe, so America would now provide security and quiet for the entire world.
...

John Stuart Mill says in his “Principles of Political Economy": “It is questionable if all the mechanical inventions yet made have lightened the day’s toil of any human being.” [1]
...

“When a labourer,” said Mr. Ashworth, a cotton magnate, to Professor Nassau W. Senior, “lays down his spade, he renders useless, for that period, a capital worth eighteen-pence. When one of our people feaves the mill, he renders useless a capital that has cost £100,000.” [69] Only fancy! making “useless” for a single moment, a capital that has cost £100,000! It is, in truth, monstrous, that a single one of our people should ever leave the factory! The increased use of machinery, as Senior after the instruction he received from Ashworth clearly perceives, makes a constantly increasing lengthening of the working-day “desirable.” [70]
...

“If,” dreamed Aristotle, the greatest thinker of antiquity, “if every tool, when summoned, or even of its own accord, could do the work that befits it, just as the creations of Daedalus moved of themselves, or the tripods of Hephaestos went of their own accord to their sacred work, if the weavers’ shuttles were to weave of themselves, then there would be no need either of apprentices for the master workers, or of slaves for the lords.”
...

“In hac urbe,” says Boxhorn (Inst. Pol., 1663), referring to the introduction of this machine into Leyden, “ante hos viginti circiter annos instrumentum quidam invenerunt textorium, quo solus plus panni et facilius conficere poterat, quan plures aequali tempore. Hinc turbae ortae et querulae textorum, tandemque usus hujus instrumenti a magistratu prohibitus est.”
...

Exactly the reasoning of the celebrated Bill Sykes. “Gentlemen of the jury, no doubt the throat of this commercial traveller has been cut. But that is not my fault, it is the fault of the knife. Must we, for such a temporary inconvenience, abolish the use of the knife? Only consider! where would agriculture and trade be without the knife? Is it not as salutary in surgery, as it is knowing in anatomy? And in addition a willing help at the festive board? If you abolish the knife — you hurl us back into the depths of barbarism.”
...

The moral degradation caused by the capitalistic exploitation of women and children has been so exhaustively depicted by F. Engels in his “Lage der Arbeitenden Klasse Englands,” and other writers, that I need only mention the subject in this place. But the intellectual desolation artificially produced by converting immature human beings into mere machines for the fabrication of surplus-value, a state of mind clearly distinguishable from that natural ignorance which keeps the mind fallow without destroying its capacity for development, its natural fertility, this desolation finally compelled even the English Parliament to make elementary education a compulsory condition to the “productive” employment of children under 14 years, in every industry subject to the Factory Acts. The spirit of capitalist production stands out clearly in the ludicrous wording of the so-called education clauses in the Factory Acts, in the absence of an administrative machinery, an absence that again makes the compulsion illusory, in the opposition of the manufacturers themselves to these education clauses, and in the tricks and dodges they put in practice for evading them.
“For this the legislature is alone to blame, by having passed a delusive law, which, while it would seem to provide that the children employed in factories shall be educated, contains no enactment by which that professed end can be secured. It provides nothing more than that the children shall on certain days of the week, and for a certain number of hours (three) in each day, be inclosed within the four walls of a place called a school, and that the employer of the child shall receive weekly a certificate to that effect signed by a person designated by the subscriber as a schoolmaster or schoolmistress.” [54]
"
]

Reference:

["
P. Chaunu;
Nominal (P. Chaunu); Verbal (“a constant…for security”); Thematic (crime; economic pressure)
"]
<br/>
["
Chaunu;
Verbal (“The revolutionary...and fear”)
"]
<br/>
["
Beccaria, Servan, Dupaty, Lacretelle, Duport, Pastoret, Target, Bergasse;
Nominal (Beccaria, Servan, Dupaty, Lacretelle, Duport, Pastoret, Target, Bergasse, Cahiers)
"]
<br/>
["
Imperialism;
Thematic (Alternative to nationalism)
"]
<br/>
["
Communism;
Thematic (the Cold War)
"]

<br/>
["
John Stuart Mill;
Nominal (John Stuart Mill, “Principles of Political Economy"); Verbal (“It is questionable…human being”)
"]

<br/>
["
Nassau W. Senior-- Mr. Ashworth;
Nominal (Nassau W. Senior); Verbal (“lays down…100,000”; “useless”; “desirable”)
"]

<br/>
["
Aristotle;
Nominal (Aristotle); Verbal (“if ever…the lords”)
"]

<br/>
["
Lancellotti—Boxhorn;
Nominal (Boxhorn); Verbal (“In hac…prohibitus est”)
"]

<br/>
["
Bill Sykes;
Nominal (Bill Sykes); Verbal (“Gentlemen of…of barbarianism”)
"]

<br/>
["
Unidentified;
Nominal ("The Factory Acts”)
"]
"""




# ["
# Petion de Villeneuve;
# 2.Contextual Explanation
# "]
# <br/>
# ["
# Seligman;
# 1.Name-dropping
# "]
# <br/>
# ["
# Joseph de Maistre;
# 3. Critical Engagement
# "]
# <br/>
# ["
# Lacretelle;
# 2. Contextual Explanation
# "]
# <br/>
# ["
# Servan;
# 4.Conceptual Application or Expansion
# "]
# <br/>

fs_ref_2 = """
If there are references proposed in parallel, please be sure to classify them as of the same type, or one holistic entity.
In these few shot examples for prompt 3, we covered all the cases. When you run the prompt, please choose the most applicable one for each reference. You don't need to identify all functions within a passage.

This is an example for your answer: 

Context:
[

The first of these, as P. Chaunu observes, was a change in the operation of economic pressures, a general rise in the standard of living, a large demographic expansion, an increase in wealth and property and 'a consequent need for security' (Chaunu, 1971, 56)...
......
One is struck, in the trials of 1782-9, by the increase in tension. There is a new severity towards the poor, a concerted rejection of evidence, a rise in mutual mistrust, hatred and fear' (Chaunu, 1966, 108).
...

Homage is paid to the 'great reformers' - Beccaria, Servan, Dupaty, Lacretelle, Duport, Pastoret, Target, Bergasse, the com­ pilers of the Cahiers, or petitions, and the Constituent Assembly - for having imposed this leniency on a legal machinery and on 'classical' theoreticians who, at the end of the eighteenth century, were still rejecting it with well-formulated arguments.
...

What is this nationalist political theory about? The nationalism I grew up with is a principled standpoint that regards the world as governed best when nations are able to chart their own independent course, cultivating their own traditions and pursuing their own interests without interference. This is opposed to imperialism, which seeks to bring peace and prosperity to the world by uniting mankind, as much as possible, under a single political regime. I do not suppose that the case for nationalism is unequivocal. Considerations can be mustered in favor of each of these theories. But what cannot be done without obfuscation is to avoid choosing between the two positions: Either you support, in principle, the ideal of an international government or regime that imposes its will on subject nations when its officials regard this as necessary; or you believe that nations should be free to set their own course in the absence of such an international government or regime. This debate between nationalism and imperialism became acutely relevant again with the fall of the Berlin Wall in 1989. At that time, the struggle against Communism ended, and the minds of Western leaders became preoccupied with two great imperialist projects: the European Union, which has progressively relieved member nations of many of the powers usually associated with political independence; and the project of establishing an American “world order,” in which nations that do not abide by international law will be coerced into doing so, principally by means of American military might. These are imperialist projects, even though their proponents do not like to call them that, for two reasons: First, their purpose is to remove decision-making from the hands of independent national governments and place it in the hands of international governments or bodies. And second, as you can immediately see from the literature produced by the individuals and institutions supporting these endeavors, they are consciously part of an imperialist political tradition, drawing their historical inspiration from the Roman Empire, the Austro-Hungarian Empire, and the British Empire. For example, Charles Krauthammer’s argument for American “Universal Dominion,” written at the dawn of the post–Cold War period, calls for America to create a “super-sovereign,” which will preside over the permanent “depreciation… of the notion of sovereignty” for all nations on earth. Krauthammer adopts the Latin term pax Americana to describe this vision, invoking the image of the United States as the new Rome: Just as the Roman Empire supposedly established a pax Romana (or “Roman peace”) that obtained security and quiet for all of Europe, so America would now provide security and quiet for the entire world.
...

John Stuart Mill says in his “Principles of Political Economy": “It is questionable if all the mechanical inventions yet made have lightened the day’s toil of any human being.” [1]
...

“When a labourer,” said Mr. Ashworth, a cotton magnate, to Professor Nassau W. Senior, “lays down his spade, he renders useless, for that period, a capital worth eighteen-pence. When one of our people feaves the mill, he renders useless a capital that has cost £100,000.” [69] Only fancy! making “useless” for a single moment, a capital that has cost £100,000! It is, in truth, monstrous, that a single one of our people should ever leave the factory! The increased use of machinery, as Senior after the instruction he received from Ashworth clearly perceives, makes a constantly increasing lengthening of the working-day “desirable.” [70]
...

“If,” dreamed Aristotle, the greatest thinker of antiquity, “if every tool, when summoned, or even of its own accord, could do the work that befits it, just as the creations of Daedalus moved of themselves, or the tripods of Hephaestos went of their own accord to their sacred work, if the weavers’ shuttles were to weave of themselves, then there would be no need either of apprentices for the master workers, or of slaves for the lords.”
...

“In hac urbe,” says Boxhorn (Inst. Pol., 1663), referring to the introduction of this machine into Leyden, “ante hos viginti circiter annos instrumentum quidam invenerunt textorium, quo solus plus panni et facilius conficere poterat, quan plures aequali tempore. Hinc turbae ortae et querulae textorum, tandemque usus hujus instrumenti a magistratu prohibitus est.”
...

Exactly the reasoning of the celebrated Bill Sykes. “Gentlemen of the jury, no doubt the throat of this commercial traveller has been cut. But that is not my fault, it is the fault of the knife. Must we, for such a temporary inconvenience, abolish the use of the knife? Only consider! where would agriculture and trade be without the knife? Is it not as salutary in surgery, as it is knowing in anatomy? And in addition a willing help at the festive board? If you abolish the knife — you hurl us back into the depths of barbarism.”
...

The moral degradation caused by the capitalistic exploitation of women and children has been so exhaustively depicted by F. Engels in his “Lage der Arbeitenden Klasse Englands,” and other writers, that I need only mention the subject in this place. But the intellectual desolation artificially produced by converting immature human beings into mere machines for the fabrication of surplus-value, a state of mind clearly distinguishable from that natural ignorance which keeps the mind fallow without destroying its capacity for development, its natural fertility, this desolation finally compelled even the English Parliament to make elementary education a compulsory condition to the “productive” employment of children under 14 years, in every industry subject to the Factory Acts. The spirit of capitalist production stands out clearly in the ludicrous wording of the so-called education clauses in the Factory Acts, in the absence of an administrative machinery, an absence that again makes the compulsion illusory, in the opposition of the manufacturers themselves to these education clauses, and in the tricks and dodges they put in practice for evading them.
“For this the legislature is alone to blame, by having passed a delusive law, which, while it would seem to provide that the children employed in factories shall be educated, contains no enactment by which that professed end can be secured. It provides nothing more than that the children shall on certain days of the week, and for a certain number of hours (three) in each day, be inclosed within the four walls of a place called a school, and that the employer of the child shall receive weekly a certificate to that effect signed by a person designated by the subscriber as a schoolmaster or schoolmistress.” [54]
"
]

Reference:
["
P. Chaunu;
2. Contextual Explanation
"]
<br/>
["
Chaunu;
2. Contextual Explanation
"]
<br/>
["
Beccaria, Servan, Dupaty, Lacretelle, Duport, Pastoret, Target, Bergasse;
1.Name-dropping
"]
<br/>
["
Imperialism;
2. Contextual Explanation
"]
<br/>
["
Communism;
1.Name-dropping
"]

<br/>
["
John Stuart Mill;
3. Critical Engagement
"]

<br/>
["
Nassau W. Senior-- Mr. Ashworth;
3.Critical Engagement
"]

<br/>
["
Aristotle;
1.Name-dropping
"]

<br/>
["
Lancellotti—Boxhorn;
2. Contextual Explanation
"]

<br/>
["
Bill Sykes;
2. Contextual Explanation
"]

<br/>
["
Unidentified;
3.Critical Engagement
"]
"""



# ["
# Petion de Villeneuve;
# Neutral
# "]
# <br/>
# ["
# Seligman;
# Neutral
# "]
# <br/>
# ["
# Joseph de Maistre;
# Positive
# "]
# <br/>
# ["
# Lacretelle;
# Negative
# "]
# <br/>
# ["
# Servan;
# Positive
# "]
# <br/>

fs_ref_3 = """
If there are references proposed in parallel, please be sure to classify them as of the same type, or one holistic entity.
In these few shot examples for prompt 4, we gave examples for all sentiments. In your application, please select the most appropriate sentiment. You don't have to find traces of all sentiments within a given passage.

This is an example for your answer: 

Context:
[

The first of these, as P. Chaunu observes, was a change in the operation of economic pressures, a general rise in the standard of living, a large demographic expansion, an increase in wealth and property and 'a consequent need for security' (Chaunu, 1971, 56)...
......
One is struck, in the trials of 1782-9, by the increase in tension. There is a new severity towards the poor, a concerted rejection of evidence, a rise in mutual mistrust, hatred and fear' (Chaunu, 1966, 108).
......

Homage is paid to the 'great reformers' - Beccaria, Servan, Dupaty, Lacretelle, Duport, Pastoret, Target, Bergasse, the com­ pilers of the Cahiers, or petitions, and the Constituent Assembly - for having imposed this leniency on a legal machinery and on 'classical' theoreticians who, at the end of the eighteenth century, were still rejecting it with well-formulated arguments.
...

What is this nationalist political theory about? The nationalism I grew up with is a principled standpoint that regards the world as governed best when nations are able to chart their own independent course, cultivating their own traditions and pursuing their own interests without interference. This is opposed to imperialism, which seeks to bring peace and prosperity to the world by uniting mankind, as much as possible, under a single political regime. I do not suppose that the case for nationalism is unequivocal. Considerations can be mustered in favor of each of these theories. But what cannot be done without obfuscation is to avoid choosing between the two positions: Either you support, in principle, the ideal of an international government or regime that imposes its will on subject nations when its officials regard this as necessary; or you believe that nations should be free to set their own course in the absence of such an international government or regime. This debate between nationalism and imperialism became acutely relevant again with the fall of the Berlin Wall in 1989. At that time, the struggle against Communism ended, and the minds of Western leaders became preoccupied with two great imperialist projects: the European Union, which has progressively relieved member nations of many of the powers usually associated with political independence; and the project of establishing an American “world order,” in which nations that do not abide by international law will be coerced into doing so, principally by means of American military might. These are imperialist projects, even though their proponents do not like to call them that, for two reasons: First, their purpose is to remove decision-making from the hands of independent national governments and place it in the hands of international governments or bodies. And second, as you can immediately see from the literature produced by the individuals and institutions supporting these endeavors, they are consciously part of an imperialist political tradition, drawing their historical inspiration from the Roman Empire, the Austro-Hungarian Empire, and the British Empire. For example, Charles Krauthammer’s argument for American “Universal Dominion,” written at the dawn of the post–Cold War period, calls for America to create a “super-sovereign,” which will preside over the permanent “depreciation… of the notion of sovereignty” for all nations on earth. Krauthammer adopts the Latin term pax Americana to describe this vision, invoking the image of the United States as the new Rome: Just as the Roman Empire supposedly established a pax Romana (or “Roman peace”) that obtained security and quiet for all of Europe, so America would now provide security and quiet for the entire world.
...

John Stuart Mill says in his “Principles of Political Economy": “It is questionable if all the mechanical inventions yet made have lightened the day’s toil of any human being.” [1]
...

“When a labourer,” said Mr. Ashworth, a cotton magnate, to Professor Nassau W. Senior, “lays down his spade, he renders useless, for that period, a capital worth eighteen-pence. When one of our people feaves the mill, he renders useless a capital that has cost £100,000.” [69] Only fancy! making “useless” for a single moment, a capital that has cost £100,000! It is, in truth, monstrous, that a single one of our people should ever leave the factory! The increased use of machinery, as Senior after the instruction he received from Ashworth clearly perceives, makes a constantly increasing lengthening of the working-day “desirable.” [70]
...

“If,” dreamed Aristotle, the greatest thinker of antiquity, “if every tool, when summoned, or even of its own accord, could do the work that befits it, just as the creations of Daedalus moved of themselves, or the tripods of Hephaestos went of their own accord to their sacred work, if the weavers’ shuttles were to weave of themselves, then there would be no need either of apprentices for the master workers, or of slaves for the lords.”
...

“In hac urbe,” says Boxhorn (Inst. Pol., 1663), referring to the introduction of this machine into Leyden, “ante hos viginti circiter annos instrumentum quidam invenerunt textorium, quo solus plus panni et facilius conficere poterat, quan plures aequali tempore. Hinc turbae ortae et querulae textorum, tandemque usus hujus instrumenti a magistratu prohibitus est.”
...

Exactly the reasoning of the celebrated Bill Sykes. “Gentlemen of the jury, no doubt the throat of this commercial traveller has been cut. But that is not my fault, it is the fault of the knife. Must we, for such a temporary inconvenience, abolish the use of the knife? Only consider! where would agriculture and trade be without the knife? Is it not as salutary in surgery, as it is knowing in anatomy? And in addition a willing help at the festive board? If you abolish the knife — you hurl us back into the depths of barbarism.”
...

The moral degradation caused by the capitalistic exploitation of women and children has been so exhaustively depicted by F. Engels in his “Lage der Arbeitenden Klasse Englands,” and other writers, that I need only mention the subject in this place. But the intellectual desolation artificially produced by converting immature human beings into mere machines for the fabrication of surplus-value, a state of mind clearly distinguishable from that natural ignorance which keeps the mind fallow without destroying its capacity for development, its natural fertility, this desolation finally compelled even the English Parliament to make elementary education a compulsory condition to the “productive” employment of children under 14 years, in every industry subject to the Factory Acts. The spirit of capitalist production stands out clearly in the ludicrous wording of the so-called education clauses in the Factory Acts, in the absence of an administrative machinery, an absence that again makes the compulsion illusory, in the opposition of the manufacturers themselves to these education clauses, and in the tricks and dodges they put in practice for evading them.
“For this the legislature is alone to blame, by having passed a delusive law, which, while it would seem to provide that the children employed in factories shall be educated, contains no enactment by which that professed end can be secured. It provides nothing more than that the children shall on certain days of the week, and for a certain number of hours (three) in each day, be inclosed within the four walls of a place called a school, and that the employer of the child shall receive weekly a certificate to that effect signed by a person designated by the subscriber as a schoolmaster or schoolmistress.” [54]
"
]

Reference:
["
P. Chaunu;
Positive
"]
<br/>
["
Chaunu;
Neutral
"]
<br/>
["
Beccaria, Servan, Dupaty, Lacretelle, Duport, Pastoret, Target, Bergasse;
Neutral
"]
<br/>
["
Imperialism;
Neutral
"]
<br/>
["
Communism;
Neutral
"]

<br/>
["
John Stuart Mill;
Negative
"]

<br/>
["
Nassau W. Senior-- Mr. Ashworth;
Positive
"]

<br/>
["
Aristotle;
Negative
"]

<br/>
["
Lancellotti—Boxhorn;
Neutral
"]

<br/>
["
Bill Sykes;
Very negative
"]

<br/>
["
Unidentified;
Strongly negative
"]
"""




# ["
# Petion de Villeneuve;
# Verbal (“the cruel…in punishing”);
# 2.Contextual Explanation;
# Neutral
# "]
# <br/>
# ["
# Seligman;
# Nominal (Seligman); Thematic (tortures);
# 1.Name-dropping;
# Neutral
# "]
# <br/>
# ["
# Joseph de Maistre;
# Nominal (Joseph de Maistre); Thematic (absolute power);
# 3. Critical Engagement;
# Positive
# "]
# <br/>
# ["
# Lachere;
# Verbal (“seeing blood flow”;“it could…with blood”);
# 2. Contextual Explanation;
# Neutral
# "]
# <br/>
# ["
# Servan;
# Verbal (“Follow one…of Empires”);
# 4.Conceptual Application or Expansion;
# Positive
# "]
# <br/>

# fs_ref_4 = """
# ["
# P. Chaunu;
# Nominal (P. Chaunu); Verbal (“a constant...for security”); Thematic (crime; economic pressure);
# 2. Contextual Explanation;
# Positive
# "]
# <br/>
# ["
# Chaunu;
# Verbal (“The revolutionary...and fear”);
# 2. Contextual Explanation;
# Neutral
# "]
# <br/>
# ["
# Beccaria, Servan, Dupaty, Lacretelle, Duport, Pastoret, Target, Bergasse;
# Nominal (Beccaria, Servan, Dupaty, Lacretelle, Duport, Pastoret, Target, Bergasse, Cahiers);
# 1.Name-dropping;
# Neutral
# "]
# <br/>
# ["
# Imperialism;
# Thematic (Alternative to nationalism);
# 2. Contextual Explanation;
# Neutral
# "]
# <br/>
# ["
# Communism;
# Thematic (the Cold War);
# 1.Name-Dropping;
# Neutral
# "]
# <br/>
# ["
# John Stuart Mill
# Nominal (John Stuart Mill, “Principles of Political Economy"); Verbal (“It is questionable…human being”)
# 3. Critical Engagement
# Negative
# "]
# <br/>
# ["
# Nassau W. Senior-- Mr. Ashworth
# Nominal (Nassau W. Senior); Verbal (“lays down…100,000”; “useless”; “desirable”)
# 3.Critical Engagement
# Positive
# "]
# <br/>
# ["
# Aristotle
# Nominal (Aristotle); Verbal (“if ever…the lords”)
# 1.Name-dropping
# Negative
# "]
# <br/>
# ["
# Lancellotti—Boxhorn
# Nominal (Boxhorn); Verbal (“In hac…prohibitus est”)
# 2. Contextual Explanation
# Neutral
# "]
# <br/>
# ["
# Bill Sykes
# Nominal (Bill Sykes); Verbal (“Gentlemen of…of barbarianism”)
# 2. Contextual Explanation
# Very negative
# "]
# <br/>
# ["
# Unidentified
# Nominal ("The Factory Acts”)
# 3.Critical Engagement
# Strongly negative
# "]
# """

In [10]:
# fs_ref = [fs_ref_0,fs_ref_1,fs_ref_2,fs_ref_3, fs_ref_4]
fs_ref = [fs_ref_0,fs_ref_1,fs_ref_2,fs_ref_3]

In [11]:
fs_arg_3 = """
For this text, you can answer this question in the following format, where you should give corresponding content in [].
Note that, you can include arbitrary amount of evidence, as the following format is just an example.

Theme: [theme1] <br/>
[This author] <br/>
1. [The referred sentence or paragraph for theme1] <br/>
2. [The referred sentence or paragraph for theme1] <br/>
...

Theme: [theme2] <br/>
[This author] <br/>
1. [The referred sentence or paragraph for theme2] <br/>
2. [The referred sentence or paragraph for theme2] <br/>
...
"""

Question

In [12]:
question_ref = ["""Prompt1:
Within the passage, please list all the references to external textual sources, including specific authors, quotes, books, ideologies, religions, and literary or philosophical schools of thoughts.
1. Please limit yourself to explicit external references.
2. Use the author’s name/the name of a group to specify each reference and list them separately; for references whose author is unidentified (like “a poet says,” “some philosophers claim”), list their authors in order as “Unidentified 1,” “Unidentified 2,” etc. For collective/unidentifiable authorship, such as the Bible, specify them by the name of the source.
3. If one external source is mentioned several times to enable the current author to make different claims, please also treat the case as multiple references and list them separately
4. If the identified reference includes a reference to another source, please list the second-order reference after the first-order one. Signify the second-order reference by putting an asterisk before it and referring to it as “author of the first-order reference—author of the second-order reference”.
Please do not explain and just give the answer!
""",

"""
Prompt2:
For each reference you identified in prompt 1, please describe its content with one or more of the following descriptions: 
1. Nominal, meaning those references that explicitly mention names of other authors, books, collections of works, and other schools of thought in the main text; for nominal references, signal their content by exact names used in the passage. Specification of authors or sources in citational practice does not count as nominal. If there are multiple nominal references, separate them by colons.
2. Verbal, meaning direct quotation of phrases and sentences from other sources; for verbal references, signal their content by abbreviated versions of the quotes that only keep the first and the last two words of the quote, with ellipses in between. If there are multiple verbal references, separate them by colons. 
3. Thematic, meaning references to others’ claims, ideas, and motifs not through direct quotes but through paraphrases; for thematic references, please signify their content by a summary in one or two philosophical terms. If there are multiple thematic references, separate them by colons. 

If there is no reference to others’ claims in a category, please give NA. 
If one external source is mentioned several times to enable the current author to make different claims, please also treat the case as multiple references and list them separately.
Lastly, formulate your answer in this way: 
Referred item: nominal (content of the nominal references); verbal (content of the verbal references); 3. thematic (content of the thematic references)
Please do not explain and just give the answer!
""",

"""
Prompt3: 
For each reference identified in prompt 1, please evaluate the intertextual function it plays by the closet descriptions below. Classify the references by “Name-Dropping,” “Contextual Explanation,” “Critical Engagement,” or “Conceptual Application or Expansion”; 

1. Name-Dropping: This category is for when the current work merely mentions the names of authors, works, or concepts as representative cases of a phenomenon or an argument, without detailed explanations that exceed one sentence. In particular, if there is a list of names whose individual significance is not discussed, please label them as “Name-Dropping.” Other markers for this category include mentioning in passing like “c.f.,” “for details, please see…,” etc. 
2. Contextual Explanation: Elements of external sources are mentioned and given some exposition to clarify the source's relevance to the author’s argument. These references add depth to the discussion but are presented without the author's personal judgment of the reference as right or wrong. Examples include references to factual evidence in support of the argument, references that intend to exemplify the author’s arguments, etc.
3. Critical Engagement: In this category, the current work actively engages with external sources by offering detailed analysis (at least one sentence of analysis for each reference) and value judgements. The author's subjective attitudes are evident as they express their agreements or disagreements with the ideas presented in these references.
4. Conceptual Application or Expansion: References that fall into this category are not only explained but are also used as a springboard for further development of the current work. The current work distills keywords or arguments from the reference and expands upon them, possibly transforming them or integrating them into a new framework. Examples include a problematic concept that is adjusted and employed in further discussion; a methodology from other sources is adopted by the current author, etc.

If one external source is mentioned several times to enable the current author to make different claims, please also treat the case as multiple references and list them separately.
Please do not explain and just give the answer!
""",

"""
Prompt4: 
Please rate the current work’s sentiment toward each reference identified in prompt 1, and characterize the sentiment in terms of strongly negative, negative, neutral, positive, strongly positive. If the author’s attitude is ambiguous or unknown, please label it as “neutral.” For references to historical facts, please label them as “neutral.” For second-order references, please assess the author’s sentiment to the second-order reference, not the sentiment of the first-order reference to the second-order reference. Please base your judgment only on the provided passage.
If one external source is mentioned several times to enable the current author to make different claims, please also treat the case as multiple references and list them separately.
"""
]
# """
# Please organize all the above answers as the following form:
# """

question_arg = [
"""
Prompt1:
Please identify all the arguments in the passage, and focus solely on the author’s contestable opinions presented in the passage. Please do not include references to other author’s arguments.
""",

"""
Prompt2:
If there are repetitions in the arguments identified before, please only keep one of them. If two or more arguments are highly congruent, please condense them into one.
""",

"""
Prompt3:
Categorize all arguments you identified in prompt2 into themes, and name each theme with one or two philosophical concepts. There can be one or multiple arguments under a theme. The same argument may appear under different themes. Please do not delete any arguments in this process. Number the argument and formulate your output as: Theme (author name: argument 1–content of the argument; argument 2–content of the argument; etc).
""",

"""
Prompt4:
There are 8 authors, i.e., Bosanquet, Brook Adams, Dewey, Emile Faguet, Jellinek, Oppenheimer, Robertson, and Russell.
Identify and cluster identical or similar themes together. Under these newly-classified themes, list the arguments from different authors as follows.

Themes<br/>
	Author 1 Name: argument(s)<br/>
	(If applicable) Author 2 Name: argument(s)<br/>
	(If applicable) Author 3 Name: argument(s)<br/>
etc.
"""
]

## QA For Arguments

In [74]:
from langchain_community.document_loaders import PyMuPDFLoader, PDFPlumberLoader

def inference(author):
    print(author)
    answers_list = []
    
    length = len(os.listdir('{}/chapters_pdf'.format(author)))

    for i in range(1, length+1):
        answers = {}
        path = "{}/chapters_pdf/{}_chapter{}.pdf".format(author, author, i)
        loader = PyMuPDFLoader(path)
        docs = loader.load()

        text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000,chunk_overlap=20)
        splits = text_splitter.split_documents(docs)
        vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings(model='text-embedding-ada-002'))
        retriever = vectorstore.as_retriever()

        template =""". Answer the question based on the following context:
        {context}

        Question: {question}
        """

        prompt = ChatPromptTemplate.from_template(template)

        llm = ChatOpenAI(model='gpt-4-0125-preview', temperature=0)

        rag_chain = (
                {"context": retriever, "question": RunnablePassthrough()}
                | prompt
                | llm
                | StrOutputParser()
        )

        for j, q in enumerate(question_arg):
            if j < 2:
                answers['{}_{}_arg_{}'.format(author, i, j)] = rag_chain.invoke(q)
            elif j == 2:
                who = 'this author is ' + author + '. '
                answers['{}_{}_arg_{}'.format(author, i, j)] = rag_chain.invoke(who+q+fs_arg_3)
        
        answers_list.append(answers)
    vectorstore.delete_collection()
    return answers_list

In [ ]:
import json
# author_list = ['Jellinek', 'Dewey']
# authors = ['Jellinek', 'Dewey','Russell']
authors = ['Test']
# authors = ['Brooks_Adams', 'emile_faguet']
# authors = ['Oppenheimer', 'robertson']


for author in authors:
    answers = inference(author)

    if not os.path.exists('{}_answers'.format(author)):
        os.makedirs('{}_answers'.format(author))
    
    for i in range(len(answers)):
        pdf_file = "{}_answers/{}_{}_arg_output.pdf".format(author, author, i+1)
        doc = SimpleDocTemplate(pdf_file, pagesize=letter)

        styles = getSampleStyleSheet()
        style = styles['Normal']
        style.wordWrap = 'CJK'  

        content = []
        for key, value in answers[i].items():
            text = f"{key}: {value}"
            paragraph = Paragraph(text, style)
            content.append(paragraph)
            content.append(Spacer(1, 12))
        doc.build(content)

## Ask the relationship

Input all answers

In [30]:
authors = ['XXX']
docs = []
answers = {}
files = []
answer_2_dict = {}
for author in authors:
    folder_path = '{}_answers/'.format(author)
    prefix = author
    files_with_prefix = glob.glob(os.path.join(folder_path, prefix + '_*'))
    for i in range(len(files_with_prefix)):
        path = "{}_answers/{}_{}_arg_output.pdf".format(author, author, i+1)
        files.append(path)
        loader = PyMuPDFLoader(path)
        docs += loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000,chunk_overlap=20)
splits = text_splitter.split_documents(docs)
vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings(model='text-embedding-ada-002'))
retriever = vectorstore.as_retriever()

template ="""Answer the question based on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

llm = ChatOpenAI(model='gpt-4-0125-preview', temperature=0)

rag_chain = (
        {"context": retriever, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
)

answers['arg_4_normal'] = rag_chain.invoke(question_arg[3])
vectorstore.delete_collection()

pdf_file = "arg_output_4.pdf"
doc = SimpleDocTemplate(pdf_file, pagesize=letter)

styles = getSampleStyleSheet()
style = styles['Normal']
style.wordWrap = 'CJK'  # 支持长文本自动换行，适用于中文等语言

# 创建内容列表
content = []
for key, value in answers.items():
    text = f"{key}: {value}"
    paragraph = Paragraph(text, style)
    content.append(paragraph)
    content.append(Spacer(1, 12))  # 在键值对之间添加空间

doc.build(content)

Input arg_2

In [ ]:
authors = ['XXX']
docs = []
answers = {}
files = []
answer_2_dict = {}
for author in authors:
    folder_path = '{}_answers/'.format(author)
    prefix = author
    files_with_prefix = glob.glob(os.path.join(folder_path, prefix + '_*'))
    for i in range(len(files_with_prefix)):
        path = "{}_answers/{}_{}_arg_output.pdf".format(author, author, i+1)
        files.append(path)
        loader = PyMuPDFLoader(path)

        for k in range(len(loader.load())):
            answer_2 = '{}_{}_arg_2'.format(author, i+1)
            if answer_2 in loader.load()[k].page_content:
                para = loader.load()[k]
                index = para.page_content.find(answer_2)
                tmp = para.page_content[index:]
                para.page_content = tmp
                docs += para

text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000,chunk_overlap=20)
splits = text_splitter.split_documents(docs)
vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings(model='text-embedding-ada-002'))
retriever = vectorstore.as_retriever()

template ="""Answer the question based on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

llm = ChatOpenAI(model='gpt-4-0125-preview', temperature=0)

rag_chain = (
        {"context": retriever, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
)

answers['arg_4_normal'] = rag_chain.invoke(question_arg[3])
vectorstore.delete_collection()

pdf_file = "arg_output_4.pdf"
doc = SimpleDocTemplate(pdf_file, pagesize=letter)

styles = getSampleStyleSheet()
style = styles['Normal']
style.wordWrap = 'CJK'  # 支持长文本自动换行，适用于中文等语言

# 创建内容列表
content = []
for key, value in answers.items():
    text = f"{key}: {value}"
    paragraph = Paragraph(text, style)
    content.append(paragraph)
    content.append(Spacer(1, 12))  # 在键值对之间添加空间

doc.build(content)

## QA For Reference

In [14]:
from langchain_community.document_loaders import PyMuPDFLoader, PDFPlumberLoader
import json
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet


def ref_inference(author):
    print(author)
    answers_list = []

    folder = "{}/paragraphs_pdf".format(author)
    if not os.path.exists('{}_answers'.format(author)):
        os.makedirs('{}_answers'.format(author))

    length = len(os.listdir(folder))

    for i in range(length):
        folder_path = '{}/paragraphs_pdf/'.format(author)
        prefix = author
        files_with_prefix = glob.glob(os.path.join(folder_path, prefix + '_' + str(i+1) + '_*'))
        print(files_with_prefix)
        for k in range(len(files_with_prefix)):
            answers = {}
#             path = folder + '/{TITLES_}{}_paragraph{}.pdf'.format(author,i+1,k+1)
            path = folder + '/{}_{}_paragraph{}.pdf'.format(author,i+1,k+1)
            if not os.path.exists(path):
                
                continue
            loader = PyMuPDFLoader(path)
            docs = loader.load()

            vectorstore = Chroma.from_documents(documents=docs, embedding=OpenAIEmbeddings(model='text-embedding-ada-002')) 
            retriever = vectorstore.as_retriever() 


            template =""". Answer the question based on the following context:
            {context}

            Question: {question}
            """
            prompt = ChatPromptTemplate.from_template(template)

            llm = ChatOpenAI(model='gpt-4-0125-preview', temperature=0)

            rag_chain = (
                    {"context": retriever, "question": RunnablePassthrough()}
                    | prompt
                    | llm
                    | StrOutputParser()
            )

            previous_answer = ''
            for j, q in enumerate(question_ref):
                if j >= 1 and j < 4:
                    answer = rag_chain.invoke(answer1 + q + fs_ref[j])
                    answers['{}_ref_FS_{}'.format(author, j)] = answer
                    previous_answer += 'The answer of prompt {}'.format(j+1) + 'is: ' + answer
                # elif j == 4:
                #     answer = rag_chain.invoke(answer1 + previous_answer + q + fs_ref[j])
                #     answers['{}_ref_FS_{}'.format(author, j)] = answer
                else:
                    answer = rag_chain.invoke(q + fs_ref[j])
                    answers['{}_ref_FS_{}'.format(author, j)] = answer
                    answer1 = 'The identified authors are: [' + answer + '].'


            pdf_file = "{}_answers/ref_output_{}_{}_{}.pdf".format(author, author, i+1, k+1)
            doc = SimpleDocTemplate(pdf_file, pagesize=letter)

            styles = getSampleStyleSheet()
            style = styles['Normal']
            style.wordWrap = 'CJK' 

         
            content = []
            for key, value in answers.items():
                text = f"{key}: {value}"
                paragraph = Paragraph(text, style)
                content.append(paragraph)
                content.append(Spacer(1, 12)) 


            doc.build(content)
            vectorstore.delete_collection()

In [ ]:
authors = ['XXX']
for i, author in enumerate(authors):
    ref_inference(author)